# Clase 03 — Extracción de datos: SQL + APIs + privacidad

**Notebook de demostración guiada para clase — Google Colab**

Este notebook acompaña la PPT de **Extracción multifuente** y prepara el terreno para el **Taller T2**.

### Ruta de la clase

```text
Base SQLite ──SQL──┐
                   ├──> Python / Pandas ──> Integración ──> Dataset
API pública ─HTTP──┘                            │
                                               └──> Desidentificación
```

> **Objetivo docente:** no aprender SQL o APIs “en profundidad”, sino entender cómo obtener datos desde distintas fuentes y convertirlos en un dataset utilizable para Ciencia de Datos.

## 0. Preparación

Este notebook es **autosuficiente**:

- crea una pequeña base SQLite automáticamente;
- no requiere subir archivos;
- usa librerías disponibles en Colab;
- intenta consultar una API real;
- si la API no responde, utiliza datos de respaldo para que la clase pueda continuar.

Ejecuta las celdas en orden.

In [1]:
import sqlite3
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Entorno preparado.")

Entorno preparado.


# Parte I — ¿De dónde vienen los datos?

Hasta ahora podríamos estar acostumbrados a:

```python
df = pd.read_csv("datos.csv")
```

Pero en un proyecto real los datos pueden venir de:

| Fuente | Ejemplo | Forma de acceso |
|---|---|---|
| Archivo | CSV / Excel | `pd.read_csv()` |
| Base de datos | SQLite / PostgreSQL | SQL |
| API | Banco Mundial | HTTP + JSON |

En esta demostración usaremos **dos fuentes distintas** y luego las integraremos.

## 1. Construimos una mini base de datos

La empresa ficticia **DataRetail LATAM** almacena información en dos tablas:

```text
CLIENTES                         VENTAS
────────                         ──────
cliente_id  ◄──────────────────  cliente_id
nombre                           venta_id
email                            fecha
pais_iso2                        monto
segmento                         canal
```

`cliente_id` es la variable que permite relacionar ambas tablas.

> **Pregunta al curso:** ¿por qué no guardar toda la información en una única tabla gigantesca?

In [2]:
clientes = pd.DataFrame({
    "cliente_id": range(101, 111),
    "nombre": [
        "Ana Torres", "Pedro Soto", "Camila Díaz", "Martín Rojas", "Lucía Pérez",
        "Diego Silva", "Valentina Mora", "Andrés López", "Sofía Castro", "Tomás Vega"
    ],
    "email": [
        "ana@example.com", "pedro@example.com", "camila@example.com", "martin@example.com",
        "lucia@example.com", "diego@example.com", "valentina@example.com",
        "andres@example.com", "sofia@example.com", "tomas@example.com"
    ],
    "pais_iso2": ["CL", "CL", "AR", "PE", "CL", "BR", "PE", "AR", "BR", "CL"],
    "segmento": ["Premium", "Regular", "Premium", "Regular", "Regular",
                 "Premium", "Regular", "Premium", "Regular", "Premium"]
})

rng = np.random.default_rng(42)

ventas = pd.DataFrame({
    "venta_id": range(1, 41),
    "cliente_id": rng.choice(clientes["cliente_id"], 40),
    "fecha": pd.date_range("2026-07-01", periods=40, freq="D").astype(str),
    "monto": rng.integers(20_000, 250_000, 40),
    "canal": rng.choice(["Web", "Tienda", "App"], 40, p=[0.45, 0.35, 0.20])
})

clientes.head()

,cliente_id,nombre,email,pais_iso2,segmento
0,101,Ana Torres,ana@example.com,CL,Premium
1,102,Pedro Soto,pedro@example.com,CL,Regular
2,103,Camila Díaz,camila@example.com,AR,Premium
3,104,Martín Rojas,martin@example.com,PE,Regular
4,105,Lucía Pérez,lucia@example.com,CL,Regular


In [3]:
ventas.head()

,venta_id,cliente_id,fecha,monto,canal
0,1,101,2026-07-01,58002,Web
1,2,108,2026-07-02,194360,App
2,3,107,2026-07-03,181120,Tienda
3,4,105,2026-07-04,101540,Web
4,5,105,2026-07-05,35621,App


In [4]:
DB_PATH = Path("ventas_demo.db")

with sqlite3.connect(DB_PATH) as conn:
    clientes.to_sql("clientes", conn, if_exists="replace", index=False)
    ventas.to_sql("ventas", conn, if_exists="replace", index=False)

print(f"Base creada: {DB_PATH.resolve()}")

Base creada: /content/ventas_demo.db


# Parte II — Consultar una base de datos con SQL

Una consulta SQL puede leerse casi como una oración:

```sql
SELECT columnas
FROM tabla
WHERE condición;
```

**SQL no es la base de datos.**  
Es el lenguaje que usamos para hacer preguntas a la base.

In [5]:
conn = sqlite3.connect(DB_PATH)

query = '''
SELECT name
FROM sqlite_master
WHERE type = 'table';
'''

pd.read_sql_query(query, conn)

,name
0,clientes
1,ventas


## 2. SELECT + FROM

Pregunta:

> **“Muéstrame las primeras cinco ventas.”**

In [6]:
query = '''
SELECT *
FROM ventas
LIMIT 5;
'''

pd.read_sql_query(query, conn)

,venta_id,cliente_id,fecha,monto,canal
0,1,101,2026-07-01,58002,Web
1,2,108,2026-07-02,194360,App
2,3,107,2026-07-03,181120,Tienda
3,4,105,2026-07-04,101540,Web
4,5,105,2026-07-05,35621,App


### SQL ↔ Pandas

Estas dos ideas son equivalentes:

```sql
SELECT *
FROM ventas
LIMIT 5;
```

```python
ventas.head()
```

El objetivo no es memorizar sintaxis todavía, sino **reconocer la operación**.

## 3. WHERE

Pregunta:

> **“¿Qué ventas fueron superiores a $150.000?”**

In [7]:
query = '''
SELECT venta_id, cliente_id, fecha, monto, canal
FROM ventas
WHERE monto > 150000
ORDER BY monto DESC;
'''

ventas_altas = pd.read_sql_query(query, conn)
ventas_altas

,venta_id,cliente_id,fecha,monto,canal
0,6,109,2026-07-06,243260,App
1,24,110,2026-07-24,242527,Tienda
2,21,106,2026-07-21,232181,Tienda
3,27,105,2026-07-27,228273,Tienda
4,8,107,2026-07-08,225417,Web
5,31,105,2026-07-31,203006,Web
6,10,101,2026-07-10,199028,Web
7,11,106,2026-07-11,194776,Web
8,2,108,2026-07-02,194360,App
9,22,104,2026-07-22,191295,Tienda


In [8]:
ventas.loc[ventas["monto"] > 150000].sort_values("monto", ascending=False).head()

,venta_id,cliente_id,fecha,monto,canal
5,6,109,2026-07-06,243260,App
23,24,110,2026-07-24,242527,Tienda
20,21,106,2026-07-21,232181,Tienda
26,27,105,2026-07-27,228273,Tienda
7,8,107,2026-07-08,225417,Web


## 4. GROUP BY

Pregunta:

> **“¿Cuánto vendemos, en promedio, por cada canal?”**

In [9]:
query = '''
SELECT
    canal,
    COUNT(*) AS n_ventas,
    ROUND(AVG(monto), 0) AS venta_promedio,
    SUM(monto) AS venta_total
FROM ventas
GROUP BY canal
ORDER BY venta_total DESC;
'''

pd.read_sql_query(query, conn)

,canal,n_ventas,venta_promedio,venta_total
0,Tienda,18,142109.0,2557963
1,Web,18,129144.0,2324585
2,App,4,149929.0,599717


In [10]:
(
    ventas
    .groupby("canal")["monto"]
    .agg(n_ventas="count", venta_promedio="mean", venta_total="sum")
    .round(0)
)

,n_ventas,venta_promedio,venta_total
canal,,,
App,4,149929.0,599717
Tienda,18,142109.0,2557963
Web,18,129144.0,2324585


## 5. JOIN — unir tablas dentro de la base

Hasta ahora conocemos el `cliente_id`, pero no sabemos su país o segmento.

Para responder preguntas que usan información de **ventas + clientes** necesitamos un `JOIN`.

```text
ventas.cliente_id  ──────  clientes.cliente_id
```

> **Pregunta al curso:** ¿qué pasaría si intentáramos unir usando `nombre` en lugar de `cliente_id`?

In [11]:
query = '''
SELECT
    v.venta_id,
    v.fecha,
    v.monto,
    v.canal,
    c.cliente_id,
    c.nombre,
    c.pais_iso2,
    c.segmento
FROM ventas AS v
JOIN clientes AS c
    ON v.cliente_id = c.cliente_id
ORDER BY v.fecha;
'''

df_ventas = pd.read_sql_query(query, conn)
df_ventas.head(8)

,venta_id,fecha,monto,canal,cliente_id,nombre,pais_iso2,segmento
0,1,2026-07-01,58002,Web,101,Ana Torres,CL,Premium
1,2,2026-07-02,194360,App,108,Andrés López,AR,Premium
2,3,2026-07-03,181120,Tienda,107,Valentina Mora,PE,Regular
3,4,2026-07-04,101540,Web,105,Lucía Pérez,CL,Regular
4,5,2026-07-05,35621,App,105,Lucía Pérez,CL,Regular
5,6,2026-07-06,243260,App,109,Sofía Castro,BR,Regular
6,7,2026-07-07,122508,Web,101,Ana Torres,CL,Premium
7,8,2026-07-08,225417,Web,107,Valentina Mora,PE,Regular


In [12]:
print("Filas:", len(df_ventas))
print("Columnas:", df_ventas.columns.tolist())
print("Países presentes:", sorted(df_ventas["pais_iso2"].unique()))

Filas: 40
Columnas: ['venta_id', 'fecha', 'monto', 'canal', 'cliente_id', 'nombre', 'pais_iso2', 'segmento']
Países presentes: ['AR', 'BR', 'CL', 'PE']


### Hasta aquí

Ya convertimos:

```text
BASE DE DATOS
    │
    │ SQL
    ▼
DataFrame de Pandas
```

Ahora veremos otra fuente completamente distinta.

# Parte III — Obtener datos desde una API

Una **API** permite que un programa solicite información a otro sistema.

```text
Python ─── GET ───> API
Python <── JSON ─── API
```

Usaremos la API pública del **Banco Mundial** para consultar información de países.

La lógica será:

1. construir una URL;
2. hacer una solicitud `GET`;
3. revisar si funcionó;
4. interpretar el JSON.

## 6. Primera solicitud GET

Consultaremos información de Chile usando su código ISO2: `CL`.

> **Antes de ejecutar:** ¿qué esperan que devuelva el servidor?

In [13]:
codigo = "CL"
url = f"https://api.worldbank.org/v2/country/{codigo}"

try:
    response = requests.get(
        url,
        params={"format": "json"},
        timeout=10
    )

    print("URL consultada:", response.url)
    print("Status code:", response.status_code)

except requests.RequestException as error:
    response = None
    print("La API no respondió:", error)

URL consultada: https://api.worldbank.org/v2/country/CL?format=json
Status code: 200


### Status codes que nos interesan

| Código | Interpretación |
|---:|---|
| 200 | OK |
| 401 / 403 | problema de autorización |
| 404 | recurso no encontrado |
| 429 | demasiadas solicitudes |
| 500 | error del servidor |

Para esta clase basta con entender que **200 significa que la solicitud fue exitosa**.

In [14]:
if response is not None and response.ok:
    datos = response.json()
    print("Tipo del objeto recibido:", type(datos))
    print("Número de elementos principales:", len(datos))
else:
    datos = None
    print("Seguiremos usando el respaldo más adelante.")

Tipo del objeto recibido: <class 'list'>
Número de elementos principales: 2


## 7. JSON

JSON representa información mediante estructuras muy parecidas a:

- listas de Python;
- diccionarios de Python.

Para la respuesta del Banco Mundial, la información del país está en:

```python
datos[1][0]
```

No necesitamos memorizar esto: normalmente se descubre leyendo la **documentación de la API** o inspeccionando la respuesta.

In [15]:
if datos is not None:
    pais = datos[1][0]

    resumen_chile = {
        "pais_iso2": codigo,
        "pais": pais["name"],
        "capital": pais["capitalCity"],
        "region": pais["region"]["value"],
        "nivel_ingreso": pais["incomeLevel"]["value"]
    }

    display(resumen_chile)
else:
    print("Sin respuesta de API en este entorno.")

{'pais_iso2': 'CL',
 'pais': 'Chile',
 'capital': 'Santiago',
 'region': 'Latin America & Caribbean ',
 'nivel_ingreso': 'High income'}

## 8. Automatizar una consulta

En Ciencia de Datos no queremos copiar y pegar una consulta por cada país.

Creamos una función que reciba:

```text
CL → información de Chile
AR → información de Argentina
PE → información de Perú
...
```

Además incluimos un pequeño **fallback** para que la demostración no dependa completamente de internet.

In [16]:
RESPALDO_PAISES = {
    "CL": {
        "pais_iso2": "CL",
        "pais": "Chile",
        "capital": "Santiago",
        "region": "Latin America & Caribbean",
        "nivel_ingreso": "High income"
    },
    "AR": {
        "pais_iso2": "AR",
        "pais": "Argentina",
        "capital": "Buenos Aires",
        "region": "Latin America & Caribbean",
        "nivel_ingreso": "Upper middle income"
    },
    "PE": {
        "pais_iso2": "PE",
        "pais": "Peru",
        "capital": "Lima",
        "region": "Latin America & Caribbean",
        "nivel_ingreso": "Upper middle income"
    },
    "BR": {
        "pais_iso2": "BR",
        "pais": "Brazil",
        "capital": "Brasilia",
        "region": "Latin America & Caribbean",
        "nivel_ingreso": "Upper middle income"
    }
}

def obtener_info_pais(codigo):
    # Consulta la API del Banco Mundial y usa respaldo si falla.
    url = f"https://api.worldbank.org/v2/country/{codigo}"

    try:
        r = requests.get(url, params={"format": "json"}, timeout=10)
        r.raise_for_status()

        payload = r.json()
        pais = payload[1][0]

        return {
            "pais_iso2": codigo,
            "pais": pais["name"],
            "capital": pais["capitalCity"],
            "region": pais["region"]["value"],
            "nivel_ingreso": pais["incomeLevel"]["value"]
        }

    except Exception as error:
        print(f"[Aviso] {codigo}: usando respaldo ({type(error).__name__})")
        return RESPALDO_PAISES.get(codigo, {
            "pais_iso2": codigo,
            "pais": None,
            "capital": None,
            "region": None,
            "nivel_ingreso": None
        })

In [17]:
obtener_info_pais("CL")

{'pais_iso2': 'CL',
 'pais': 'Chile',
 'capital': 'Santiago',
 'region': 'Latin America & Caribbean ',
 'nivel_ingreso': 'High income'}

## 9. De API a DataFrame

Ahora obtenemos información solo para los países que realmente aparecen en nuestras ventas.

In [18]:
codigos_paises = sorted(df_ventas["pais_iso2"].unique())
codigos_paises

['AR', 'BR', 'CL', 'PE']

In [19]:
info_paises = [obtener_info_pais(codigo) for codigo in codigos_paises]

df_paises = pd.DataFrame(info_paises)
df_paises

,pais_iso2,pais,capital,region,nivel_ingreso
0,AR,Argentina,Buenos Aires,Latin America & Caribbean,Upper middle income
1,BR,Brazil,Brasilia,Latin America & Caribbean,Upper middle income
2,CL,Chile,Santiago,Latin America & Caribbean,High income
3,PE,Peru,Lima,Latin America & Caribbean,Upper middle income


# Parte IV — Integración multifuente

Ahora tenemos:

```text
df_ventas                       df_paises
─────────                       ──────────
venta_id                        pais_iso2
monto                           pais
canal                           capital
pais_iso2  ◄──────────────────► region
segmento                        nivel_ingreso
```

La clave común es **`pais_iso2`**.

En SQL usamos `JOIN`.  
En Pandas podemos usar `merge()`.

In [20]:
df_integrado = df_ventas.merge(
    df_paises,
    on="pais_iso2",
    how="left"
)

df_integrado.head()

,venta_id,fecha,monto,canal,cliente_id,nombre,pais_iso2,segmento,pais,capital,region,nivel_ingreso
0,1,2026-07-01,58002,Web,101,Ana Torres,CL,Premium,Chile,Santiago,Latin America & Caribbean,High income
1,2,2026-07-02,194360,App,108,Andrés López,AR,Premium,Argentina,Buenos Aires,Latin America & Caribbean,Upper middle income
2,3,2026-07-03,181120,Tienda,107,Valentina Mora,PE,Regular,Peru,Lima,Latin America & Caribbean,Upper middle income
3,4,2026-07-04,101540,Web,105,Lucía Pérez,CL,Regular,Chile,Santiago,Latin America & Caribbean,High income
4,5,2026-07-05,35621,App,105,Lucía Pérez,CL,Regular,Chile,Santiago,Latin America & Caribbean,High income


## 10. Validar después del merge

Integrar datos no termina con ejecutar `merge()`.

Siempre deberíamos revisar, como mínimo:

- ¿se conservaron las filas esperadas?
- ¿aparecieron valores faltantes?
- ¿hubo duplicaciones inesperadas?

In [21]:
print("Filas antes del merge:", len(df_ventas))
print("Filas después del merge:", len(df_integrado))
print("\nValores faltantes:")
display(df_integrado.isna().sum())

Filas antes del merge: 40
Filas después del merge: 40

Valores faltantes:


,0
venta_id,0
fecha,0
monto,0
canal,0
cliente_id,0
nombre,0
pais_iso2,0
segmento,0
pais,0
capital,0


In [22]:
(
    df_integrado
    .groupby(["region", "segmento"])["monto"]
    .agg(["count", "mean", "sum"])
    .round(0)
)

count      mean      sum
region                     segmento                          
Latin America & Caribbean  Premium      21  128411.0  2696622
                           Regular      19  146613.0  2785643

> **Pregunta al curso:** ¿qué ganamos al integrar la API si todas nuestras ventas ya tenían un código de país?

La integración **enriquece** el dataset: ahora podemos formular preguntas que requieren contexto externo.

# Parte V — Privacidad y minimización de datos

Nuestro dataset todavía contiene:

- `nombre`;
- `cliente_id`.

Si nuestro objetivo es analizar ventas por país, canal o segmento:

> **¿necesitamos realmente identificar a Ana, Pedro o Camila?**

Principio útil:

### conservar solo los datos necesarios para el propósito analítico

In [23]:
df_integrado[[
    "cliente_id", "nombre", "monto",
    "pais", "segmento", "canal"
]].head()

,cliente_id,nombre,monto,pais,segmento,canal
0,101,Ana Torres,58002,Chile,Premium,Web
1,108,Andrés López,194360,Argentina,Premium,App
2,107,Valentina Mora,181120,Peru,Regular,Tienda
3,105,Lucía Pérez,101540,Chile,Regular,Web
4,105,Lucía Pérez,35621,Chile,Regular,App


## 11. Pseudonimización

Podemos reemplazar un identificador por un código pseudónimo.

**Importante:** esto no garantiza anonimización completa.

Para esta clase hablaremos de:

> **dataset desidentificado / pseudonimizado**

In [24]:
SECRETO = "CD2026_DEMO"

def pseudonimizar(valor):
    texto = f"{SECRETO}|{valor}"
    return hashlib.sha256(texto.encode()).hexdigest()[:12]

df_integrado["cliente_anon"] = (
    df_integrado["cliente_id"]
    .apply(pseudonimizar)
)

df_integrado[["cliente_id", "cliente_anon"]].head()

,cliente_id,cliente_anon
0,101,3aaaaababbd6
1,108,7461bdc242d6
2,107,2b535ea3b381
3,105,e94915827733
4,105,e94915827733


In [25]:
df_final = df_integrado.drop(
    columns=["cliente_id", "nombre"]
)

df_final.head()

,venta_id,fecha,monto,canal,pais_iso2,segmento,pais,capital,region,nivel_ingreso,cliente_anon
0,1,2026-07-01,58002,Web,CL,Premium,Chile,Santiago,Latin America & Caribbean,High income,3aaaaababbd6
1,2,2026-07-02,194360,App,AR,Premium,Argentina,Buenos Aires,Latin America & Caribbean,Upper middle income,7461bdc242d6
2,3,2026-07-03,181120,Tienda,PE,Regular,Peru,Lima,Latin America & Caribbean,Upper middle income,2b535ea3b381
3,4,2026-07-04,101540,Web,CL,Regular,Chile,Santiago,Latin America & Caribbean,High income,e94915827733
4,5,2026-07-05,35621,App,CL,Regular,Chile,Santiago,Latin America & Caribbean,High income,e94915827733


## 12. Validaciones finales

Un pipeline reproducible debería comprobar automáticamente algunas condiciones mínimas.

In [26]:
assert len(df_final) == len(df_ventas)
assert "nombre" not in df_final.columns
assert "cliente_id" not in df_final.columns
assert "cliente_anon" in df_final.columns
assert df_final["region"].notna().all()

print("✓ Validaciones superadas")
print("Dimensión final:", df_final.shape)

✓ Validaciones superadas
Dimensión final: (40, 11)


# Cierre — el pipeline completo

```text
SQLite
  │
  │ SELECT / WHERE / GROUP BY / JOIN
  ▼
df_ventas
  │
  │                         API Banco Mundial
  │                                │
  │                                │ GET + JSON
  │                                ▼
  │                            df_paises
  │                                │
  └────────────── merge ───────────┘
                  │
                  ▼
             df_integrado
                  │
           minimización +
          pseudonimización
                  │
                  ▼
              df_final
```

### Lo importante de hoy

1. **SQL** permite consultar datos almacenados en bases relacionales.
2. Una **API** permite solicitar datos a otro sistema.
3. **JSON** es un formato habitual de respuesta.
4. `pandas.merge()` permite integrar fuentes.
5. Después de integrar, hay que **validar**.
6. Antes de guardar o compartir, hay que revisar **privacidad y necesidad de cada variable**.

## Antes del Taller T2

En el taller ustedes harán el mismo proceso, pero con menos código entregado:

- inspeccionar una base SQLite;
- completar una consulta con `JOIN`;
- consultar la API;
- transformar la respuesta;
- integrar las dos fuentes;
- desidentificar;
- generar el dataset final.


In [27]:
df_final.to_csv("dataset_demo_final.csv", index=False)
print("Archivo generado: dataset_demo_final.csv")

Archivo generado: dataset_demo_final.csv
